In [1]:
import pandas as pd
import numpy as np
import warnings, time
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression 
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit, RandomizedSearchCV
from joblib import dump

warnings.filterwarnings('ignore', category=UserWarning, module='xgboost')
warnings.filterwarnings('ignore', category=Warning)

print("--- [Início do Pipeline (v33 - Otimização Temporal Melhorada)] ---")
print("A otimizar (GridSearch) o Stacking (v33+) usando TimeSeriesSplit...")

# --- 1. FUNÇÕES AUXILIARES ---
def formatar_celula(series_coluna):
    s = series_coluna.astype(str).replace('NULL', pd.NA)
    s = s.str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
    s = s.str.upper().str.replace(r'[^A-Z0-9]+', '_', regex=True).str.strip('_')
    s = s.replace('', pd.NA)
    return s

def preprocessar_dados(df, colunas_scaler_treinadas=None, scaler=None):
    df_final_row_ids = np.arange(1, len(df) + 1) if 'AVERAGE_SPEED_DIFF' not in df.columns else None

    if 'AVERAGE_CLOUDINESS' in df.columns:
        df['AVERAGE_CLOUDINESS'] = formatar_celula(df['AVERAGE_CLOUDINESS']).replace('NONE', 'NONE').fillna('NONE')

    if 'AVERAGE_RAIN' in df.columns:
        df['AVERAGE_RAIN'] = formatar_celula(df['AVERAGE_RAIN'])

    record_date_col = pd.to_datetime(df['record_date'], errors='coerce', dayfirst=True) if 'record_date' in df.columns else None

    if record_date_col is not None:
        df['Hora_sin'] = np.sin(2 * np.pi * record_date_col.dt.hour / 24)
        df['Hora_cos'] = np.cos(2 * np.pi * record_date_col.dt.hour / 24)
        df['Mes_sin'] = np.sin(2 * np.pi * record_date_col.dt.month / 12)
        df['Mes_cos'] = np.cos(2 * np.pi * record_date_col.dt.month / 12)
        df['DIA_SEMANA'] = record_date_col.dt.dayofweek
        df['IS_WEEKEND'] = (df['DIA_SEMANA'] >= 5).astype(int)
        df['IS_RUSH_HOUR'] = record_date_col.dt.hour.isin([7,8,9,17,18,19]).astype(int)

    df = df.drop(columns=[c for c in ['city_name','AVERAGE_RAIN','AVERAGE_PRECIPITATION','record_date'] if c in df.columns], errors='ignore')

    # One-hot encoding direto
    for col in ['LUMINOSITY', 'AVERAGE_CLOUDINESS', 'DIA_SEMANA']:
        if col in df.columns:
            df = pd.get_dummies(df, columns=[col], prefix=col[:3].upper() if col != 'DIA_SEMANA' else 'DAY', dtype=int)

    cols_norm = [
        'AVERAGE_FREE_FLOW_SPEED','AVERAGE_TIME_DIFF','AVERAGE_FREE_FLOW_TIME',
        'AVERAGE_TEMPERATURE','AVERAGE_ATMOSP_PRESSURE','AVERAGE_HUMIDITY',
        'AVERAGE_WIND_SPEED','IS_WEEKEND','IS_RUSH_HOUR',
        'Hora_sin','Hora_cos','Mes_sin','Mes_cos'
    ]
    cols_existentes = [c for c in cols_norm if c in df.columns]

    if scaler is None:
        scaler = MinMaxScaler()
        if cols_existentes:
            df[cols_existentes] = scaler.fit_transform(df[cols_existentes])
        return df, scaler, cols_existentes, df_final_row_ids
    else:
        if cols_existentes:
            df[cols_existentes] = scaler.transform(df[cols_existentes])
        return df, scaler, cols_existentes, df_final_row_ids


# --- 2. CARREGAR E ORDENAR DADOS ---
print("\n--- [Passo 1/6] A carregar e ORDENAR dados de Treino... ---")
try:
    df_train = pd.read_csv("datasets/training_data.csv", sep=None, engine='python', encoding='latin-1')
except Exception:
    df_train = pd.read_csv("datasets/training_data.csv", sep=',', encoding='latin-1')

if 'record_date' in df_train.columns:
    df_train['record_date'] = pd.to_datetime(df_train['record_date'], dayfirst=True, errors='coerce')
    df_train = df_train.sort_values('record_date').reset_index(drop=True)

if 'AVERAGE_SPEED_DIFF' not in df_train.columns:
    raise KeyError(f"Coluna 'AVERAGE_SPEED_DIFF' não encontrada! Colunas lidas: {list(df_train.columns)}")

print("... Dados de treino ordenados.")

# --- LABEL ENCODER ---
y_train_raw = df_train['AVERAGE_SPEED_DIFF']
le = LabelEncoder()
y_train_encoded = le.fit_transform(formatar_celula(y_train_raw).fillna('NONE'))
print(f"Classes do Alvo: {list(le.classes_)}")

# --- 3. PRÉ-PROCESSAR TREINO ---
print("--- [Passo 2/6] A pré-processar os dados de TREINO ORDENADOS... ---")
X_train, scaler_final, colunas_scaler_final, _ = preprocessar_dados(df_train.drop(columns=['AVERAGE_SPEED_DIFF']))
X_train = X_train.fillna(0)
print(f"Shape final do Treino: {X_train.shape}")

# --- 4. CONFIGURAR STACKING + GRIDSEARCH ---
print("--- [Passo 3/6] A configurar Stacking e GridSearchCV Temporal... ---")

base_xgb = XGBClassifier(
    subsample=0.8, colsample_bytree=0.8, 
    use_label_encoder=False, eval_metric='mlogloss',
    random_state=42, n_jobs=-1, verbosity=0
)
base_lgbm = LGBMClassifier(
    subsample=0.9, colsample_bytree=0.8, 
    random_state=42, n_jobs=-1, verbose=-1
)
meta_lr = LogisticRegression(random_state=42, max_iter=2000, n_jobs=-1)

stack_clf = StackingClassifier(
    estimators=[('xgb', base_xgb), ('lgbm', base_lgbm)],
    final_estimator=meta_lr, cv=5, n_jobs=-1
)

param_grid = {
    'xgb__learning_rate': [0.05],
    'xgb__max_depth': [5],
    'xgb__n_estimators': [100],
    'lgbm__learning_rate': [0.01],
    'lgbm__num_leaves': [40],
    'lgbm__n_estimators': [200],
    'final_estimator__C': [1.0]
}


tscv = TimeSeriesSplit(n_splits=5)

grid_search_ts = RandomizedSearchCV(
    stack_clf,
    param_distributions=param_grid,
    n_iter=10,  # testa 10 combinações aleatórias
    cv=tscv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1,
    random_state=42
)


# --- 5. EXECUTAR GRIDSEARCH ---
print("\n--- [Passo 4/6] A EXECUTAR GridSearchCV com TimeSeriesSplit... ---")
start_time = time.time()
grid_search_ts.fit(X_train, y_train_encoded)
end_time = time.time()

print(f"\n✅ Otimização concluída em {(end_time - start_time)/60:.2f} min")
print(f"Melhor score: {grid_search_ts.best_score_:.5f}")
print("Melhores parâmetros:", grid_search_ts.best_params_)

best_stack = grid_search_ts.best_estimator_

# --- 6. TESTE E SUBMISSION ---
print("\n--- [Passo 5/6] A processar dados de Teste e gerar submission... ---")
df_test = pd.read_csv("datasets/test_data.csv", sep=',', encoding='latin-1')
df_test['record_date'] = pd.to_datetime(df_test.get('record_date', pd.NaT), errors='coerce', dayfirst=True)

X_test, _, _, test_ids = preprocessar_dados(df_test, colunas_scaler_treinadas=colunas_scaler_final, scaler=scaler_final)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# Previsões finais
best_stack.fit(X_train, y_train_encoded)
preds = le.inverse_transform(best_stack.predict(X_test))
preds = pd.Series(preds).str.title()

submission = pd.DataFrame({'RowId': np.arange(1, len(preds)+1), 'Speed_Diff': preds})
submission.to_csv('submission_v33_Optimized.csv', index=False)
print("✅ Ficheiro 'submission_v33_Optimized.csv' criado com sucesso!")

# --- (Opcional) Guardar modelo final para reutilizar ---
dump(best_stack, 'best_stack_model_v33.joblib')
print("Modelo final guardado em 'best_stack_model_v33.joblib'.")

print("\n--- [FIM DO PIPELINE OTIMIZADO] ---")
print(f"Melhor score temporal: {grid_search_ts.best_score_:.5f}")


--- [Início do Pipeline (v33 - Otimização Temporal Melhorada)] ---
A otimizar (GridSearch) o Stacking (v33+) usando TimeSeriesSplit...

--- [Passo 1/6] A carregar e ORDENAR dados de Treino... ---
... Dados de treino ordenados.
Classes do Alvo: ['HIGH', 'LOW', 'MEDIUM', 'NAN', 'VERY_HIGH']
--- [Passo 2/6] A pré-processar os dados de TREINO ORDENADOS... ---
Shape final do Treino: (6812, 33)
--- [Passo 3/6] A configurar Stacking e GridSearchCV Temporal... ---

--- [Passo 4/6] A EXECUTAR GridSearchCV com TimeSeriesSplit... ---
Fitting 5 folds for each of 1 candidates, totalling 5 fits

✅ Otimização concluída em 2.96 min
Melhor score: 0.77956
Melhores parâmetros: {'xgb__n_estimators': 100, 'xgb__max_depth': 5, 'xgb__learning_rate': 0.05, 'lgbm__num_leaves': 40, 'lgbm__n_estimators': 200, 'lgbm__learning_rate': 0.01, 'final_estimator__C': 1.0}

--- [Passo 5/6] A processar dados de Teste e gerar submission... ---
✅ Ficheiro 'submission_v33_Optimized.csv' criado com sucesso!
Modelo final guard